# Constructing the Leslie matrix. Dominant eigenvalue. SAD

In [2]:
import pandas as pd
import numpy as np

# -----------------------------
# Parameters
# -----------------------------
smooth_window = 5   # centered rolling mean window
eps = 1e-6          # small constant to avoid division by zero
excel_file = "data.xlsx"  # input file

# -----------------------------
# Read data from Excel
# -----------------------------
allfem_counts = pd.read_excel(excel_file, sheet_name="AllFemales", index_col=0)
mature_counts = pd.read_excel(excel_file, sheet_name="MatureFemales", index_col=0)
tow_counts = pd.read_excel(excel_file, sheet_name="TowCount", index_col=0)

allfem_counts.index = allfem_counts.index.astype(int)
mature_counts.index = mature_counts.index.astype(int)
tow_counts.index = tow_counts.index.astype(int)

# -----------------------------
# Compute CPUE
# -----------------------------
allfem_df = allfem_counts.div(tow_counts["Tows"], axis=0)
mature_df = mature_counts.div(tow_counts["Tows"], axis=0)

# Standardize column names
age_cols = [f"Age{i}" for i in range(1, allfem_df.shape[1]+1)]
allfem_df.columns = age_cols
mature_df.columns = age_cols

# -----------------------------
# Recruitment (Age1 next year)
# -----------------------------
recruit_df = allfem_df[["Age1"]].rename(columns={"Age1":"Recruitment"}).copy()
recruit_df["Year"] = recruit_df.index - 1
recruit_df.set_index("Year", inplace=True)

# Merge mature females with recruitment
df = mature_df.merge(recruit_df, left_index=True, right_index=True, how="inner")

# Normalize mature counts into proportions
df_prop = df[age_cols].div(df[age_cols].sum(axis=1), axis=0)

# Scale proportions by recruitment
df_scaled = df_prop.mul(df["Recruitment"], axis=0)

# -----------------------------
# Compute fecundities (geometric mean)
# -----------------------------
fecundity_final = {}
for age in age_cols:
    vals = df_scaled[age].values
    vals = vals[vals>0]
    if len(vals)>0:
        gm = np.exp(np.mean(np.log(vals)))
    else:
        gm = 0.0
    fecundity_final[age] = round(gm,3)

# Apply biological rule: Age1 fecundity = 0
fecundity_final["Age1"] = 0.0

# -----------------------------
# Compute survival probabilities S_i from smoothed all-females CPUE
# -----------------------------
smoothed_df = allfem_df.rolling(window=smooth_window, center=True, min_periods=1).mean()
ages = [1,2,3,4,5]  # only ages 1-5 for survival to next age
valid_t_years = [y for y in allfem_df.index if (y+1) in allfem_df.index]

r_dict = {i: [] for i in ages}
for t in valid_t_years:
    t1 = t+1
    for i in ages:
        cpue_i_t = smoothed_df.at[t, f"Age{i}"]
        cpue_ip1_t1 = smoothed_df.at[t1, f"Age{i+1}"]
        r_dict[i].append(cpue_ip1_t1/(cpue_i_t + eps))

S = {}
for i in ages:
    vals = np.array(r_dict[i])
    pos = vals[vals>0]
    if len(pos)>0:
        gm = np.exp(np.mean(np.log(pos)))
        S_i = min(1.0, gm)
    else:
        S_i = 0.0
    S[f"S_{i}"] = round(S_i,4)

# Last age survival (Age6) = 0
S["S_6"] = 0.0

# -----------------------------
# Build Leslie matrix
# -----------------------------
F = [fecundity_final[f"Age{i}"] for i in range(1,7)]
L = np.zeros((6,6))
L[0,:] = F
for i in range(1,6):
    L[i, i-1] = S[f"S_{i}"]
L[5,5] = S["S_6"]

# -----------------------------
# Dominant eigenvalue and stable age distribution
# -----------------------------
eigvals, eigvecs = np.linalg.eig(L)
lambda1 = max(eigvals.real)
i_max = np.argmax(eigvals.real)
stable_dist = eigvecs[:, i_max].real
stable_dist /= stable_dist.sum()  # normalize

# -----------------------------
# Compute observed average distributions
# -----------------------------
def compute_observed_dist(df, start_year, end_year):
    df_period = df.loc[start_year:end_year, age_cols]
    avg_counts = df_period.mean(axis=0)
    obs_dist = avg_counts / avg_counts.sum()
    return obs_dist

obs_1992_2002 = compute_observed_dist(allfem_df, 1992, 2002)
obs_1992_2023 = compute_observed_dist(allfem_df, 1992, 2023)

# RMSE function
def rmse(a, b):
    return np.sqrt(np.mean((np.array(a) - np.array(b))**2))

rmse_1992_2002 = rmse(stable_dist, obs_1992_2002)
rmse_1992_2023 = rmse(stable_dist, obs_1992_2023)

# -----------------------------
# Print results side-by-side
# -----------------------------
print("\n=== Age Distributions (Proportions) ===")
print(f"{'Age':>5} | {'SAD':>6} | {'Obs 92-02':>10} | {'Obs 92-23':>10}")
print("-"*40)
for i, age in enumerate(age_cols):
    print(f"{age:>5} | {stable_dist[i]:6.3f} | {obs_1992_2002[i]:10.3f} | {obs_1992_2023[i]:10.3f}")

print(f"\nRMSE vs SAD (1992-2002): {rmse_1992_2002:.4f}")
print(f"RMSE vs SAD (1992-2023): {rmse_1992_2023:.4f}")

# -----------------------------
# Print fecundities, survival, Leslie matrix, lambda1
# -----------------------------
print("\n=== Estimated Fecundities ===")
for age in age_cols:
    print(f"{age}: {fecundity_final[age]}")

print("\n=== Estimated Survival Probabilities ===")
for key, val in S.items():
    print(f"{key}: {val}")

print("\nLeslie Matrix (6x6):")
print(np.round(L,3))
print(f"\nDominant Eigenvalue (λ1): {lambda1:.4f}")



=== Age Distributions (Proportions) ===
  Age |    SAD |  Obs 92-02 |  Obs 92-23
----------------------------------------
 Age1 |  0.167 |      0.195 |      0.235
 Age2 |  0.165 |      0.177 |      0.195
 Age3 |  0.178 |      0.179 |      0.180
 Age4 |  0.216 |      0.209 |      0.190
 Age5 |  0.141 |      0.132 |      0.112
 Age6 |  0.133 |      0.108 |      0.089

RMSE vs SAD (1992-2002): 0.0168
RMSE vs SAD (1992-2023): 0.0387

=== Estimated Fecundities ===
Age1: 0.0
Age2: 0.036
Age3: 0.204
Age4: 0.255
Age5: 0.144
Age6: 0.118

=== Estimated Survival Probabilities ===
S_1: 0.7893
S_2: 0.8595
S_3: 0.9714
S_4: 0.5227
S_5: 0.7504
S_6: 0.0

Leslie Matrix (6x6):
[[0.    0.036 0.204 0.255 0.144 0.118]
 [0.789 0.    0.    0.    0.    0.   ]
 [0.    0.86  0.    0.    0.    0.   ]
 [0.    0.    0.971 0.    0.    0.   ]
 [0.    0.    0.    0.523 0.    0.   ]
 [0.    0.    0.    0.    0.75  0.   ]]

Dominant Eigenvalue (λ1): 0.7985
